# Introduction

This Notebook implements a user-based collaborative filtering recommender system.

# Data preparation

In [1]:
import os
import torch
import pandas as pd
import torch.nn.functional as F

In [2]:
def load_ratings(path):
    rows = []

    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            user_id, movie_id, rating, timestamp = line.strip().split("::")

            rows.append({
                "user_id": int(user_id) - 1,
                "movie_id": int(movie_id) - 1,
                "rating": float(rating)
            })

    return pd.DataFrame(rows)


def load_movies(path):
    movies = {}

    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("::")

            movie_id = int(parts[0])
            title = parts[1]

            movies[movie_id - 1] = title

    return pd.DataFrame.from_dict(
        movies,
        orient="index",
        columns=["title"]
    )

In [3]:
root_path = "/kaggle/input/datasets/sherinclaudia/movielens"
ratings_path = os.path.join(root_path, "ratings.dat")
movies_path = os.path.join(root_path, "movies.dat")

In [4]:
ratings_df = load_ratings(ratings_path)
movies_df = load_movies(movies_path)
movies_df = movies_df.reset_index()
movies_df.rename(columns={'index': 'movie_id'}, inplace=True)

# User-item matrix

In [5]:
user_item_df = ratings_df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating",
    fill_value=0
)

ratings = torch.tensor(
    user_item_df.values,
    dtype=torch.float32
)

print(f"Ratings shape: {ratings.shape}")

Ratings shape: torch.Size([6040, 3706])


For user-based collaborative filtering, we compare users, not movies.

In [6]:
# Normalize each user vector
normalized_users = F.normalize(ratings, p=2, dim=1)

# User-user cosine similarity matrix
user_similarity = normalized_users @ normalized_users.T

print(f"User similarity: {user_similarity.shape}")

User similarity: torch.Size([6040, 6040])


# Recommendation function

In [7]:
def recommend_user_based(
    target_user_id,
    ratings,
    user_similarity,
    movies_df,
    user_item_df,
    top_k_users=20,
    top_k_movies=10
):
    target_user_idx = target_user_id - 1

    similarities = user_similarity[target_user_idx].clone()

    # Do not compare the user with themselves
    similarities[target_user_idx] = 0

    # Keep only the most similar users
    top_users = torch.topk(similarities, top_k_users).indices
    top_similarities = similarities[top_users]

    neighbor_ratings = ratings[top_users]

    # Weighted average of neighbor ratings
    predicted_scores = top_similarities @ neighbor_ratings
    predicted_scores = predicted_scores / torch.clamp(
        top_similarities.sum(),
        min=1e-8
    )

    # Do not recommend movies the target user already rated
    already_rated = ratings[target_user_idx] > 0
    predicted_scores[already_rated] = -1

    top_movie_positions = torch.topk(
        predicted_scores,
        top_k_movies
    ).indices.tolist()

    # Convert matrix column positions back to movie IDs
    movie_ids = [
        user_item_df.columns[pos]
        for pos in top_movie_positions
    ]

    recommendations = movies_df.loc[movie_ids].copy()
    recommendations["predicted_score"] = [
        predicted_scores[pos].item()
        for pos in top_movie_positions
    ]

    return recommendations

Let's test it for one user.

In [8]:
recommendations = recommend_user_based(
    target_user_id=1,
    ratings=ratings,
    user_similarity=user_similarity,
    movies_df=movies_df,
    user_item_df=user_item_df,
    top_k_users=20,
    top_k_movies=10
)

print(recommendations)

      movie_id                               title  predicted_score
2080      2148   House II: The Second Story (1987)         3.925968
2077      2145              St. Elmo's Fire (1985)         3.141966
2095      2163          Surf Nazis Must Die (1987)         3.116527
363        366                    Mask, The (1994)         2.988083
1281      1300             Forbidden Planet (1956)         2.912254
2084      2152                Avengers, The (1998)         2.500913
2086      2154  Slums of Beverly Hills, The (1998)         2.492589
1209      1226  Once Upon a Time in America (1984)         2.441148
479        482             King of the Hill (1993)         2.421509
2079      2147                        House (1986)         2.405895
